# 16b — UR5e state estimation with moving horizons

**Outcomes:** derive encoder-coordinate dynamics, build a constrained MHE,
combine encoder and tool-position observations, and compare velocity estimates
with numerical differentiation. This standalone notebook repeats the calibration
needed by its observation model. See [16a](16a_ur5e_calibration.ipynb) for the
calibration experiment and [17](17_ur5e_ocp.ipynb) for feedback control.

### Prepare the notebook and optional interactive frontend

Imports below provide array operations, CasADi graphs, and presentation only.
Every robot model and optimization formulation is defined in this notebook.
Plotly gives hover/zoom plots when installed; Matplotlib is the fallback.
Widgets use explicit **Run** and **Reset** buttons, so moving a slider cannot
launch a solve. `CASADI_TUTORIAL_WIDGETS=0` disables widgets for batch execution.

In [ ]:
import os
import sys
from pathlib import Path
from time import perf_counter

repository_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import casadi as ca
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from casadi_tutorial.interactive import experiment
from casadi_tutorial.plotting import display_plot, plot_robot, plot_series
from casadi_tutorial.diagnostics import bound_report, max_bound_violation, solver_summary

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

np.set_printoptions(precision=5, suppress=True)

### Write the robot model: frames, parameters, and coordinates

The table below is the [UR5e standard DH geometry published by Universal Robots](https://www.universal-robots.com/articles/ur/application-installation/dh-parameters-for-calculations-of-kinematics-and-dynamics/).
Rows follow joints 1–6 from base to tool; all lengths are metres and all angles are radians.

| Joint | $a_i$ | $d_i$ | $\alpha_i$ |
|---|---:|---:|---:|
| 1 | 0 | 0.1625 | $\pi/2$ |
| 2 | −0.425 | 0 | 0 |
| 3 | −0.3922 | 0 | 0 |
| 4 | 0 | 0.1333 | $\pi/2$ |
| 5 | 0 | 0.0997 | $-\pi/2$ |
| 6 | 0 | 0.0996 | 0 |

For **standard** DH, the relative transform is

$$
{}^{i-1}T_i=\begin{bmatrix}
c_i&-s_i c_{\alpha_i}&s_i s_{\alpha_i}&a_i c_i\\
s_i&c_i c_{\alpha_i}&-c_i s_{\alpha_i}&a_i s_i\\
0&s_{\alpha_i}&c_{\alpha_i}&d_i\\0&0&0&1
\end{bmatrix},\qquad
T^0_6=\prod_{i=1}^6{}^{i-1}T_i=\begin{bmatrix}R&p\\0&1\end{bmatrix}.
$$

Calibration will adjust only $\theta=[\Delta a_2,\Delta a_3,\delta_2,\delta_3]$.
Here $q$ always means the six **encoder coordinates**, and
$q_{\rm physical}=q+[0,\delta_2,\delta_3,0,0,0]^T$.
The offset is added exactly once, inside this model. Fixed base and tool frames are assumed known.
`pose_map(q, theta)` returns $4\times4$; `position_map` returns $3\times1$;
`origins_map` returns $3\times7$. Joint, speed, and acceleration limits below are conservative teaching choices.
These functions are defined locally so the whole graph can be inspected and edited here.

In [ ]:
# Standard DH, metres and radians. The table is included here rather than imported.
dh_a = np.array([0.0, -0.425, -0.3922, 0.0, 0.0, 0.0])
dh_d = np.array([0.1625, 0.0, 0.0, 0.1333, 0.0997, 0.0996])
dh_alpha = np.array([np.pi / 2, 0.0, 0.0, np.pi / 2, -np.pi / 2, 0.0])
q_min_physical = np.full(6, -2 * np.pi)
q_max_physical = np.full(6, 2 * np.pi)
v_max = np.full(6, np.pi)
u_max = np.full(6, 2 * np.pi)


def encoder_bias(theta):
    """theta = [delta_a2, delta_a3, delta_encoder2, delta_encoder3]."""
    return ca.vertcat(0, theta[2], theta[3], 0, 0, 0)


def dh_transform(angle, distance, length, twist):
    """One STANDARD-DH transform RotZ(angle) TransZ(d) TransX(a) RotX(alpha)."""
    c, s = ca.cos(angle), ca.sin(angle)
    ct, st = np.cos(twist), np.sin(twist)
    return ca.vertcat(
        ca.horzcat(c, -s * ct, s * st, length * c),
        ca.horzcat(s, c * ct, -c * st, length * s),
        ca.horzcat(0, st, ct, distance),
        ca.horzcat(0, 0, 0, 1),
    )


def link_transforms(q_encoder, theta=None):
    """Seven base-to-frame transforms, including the identity base frame."""
    theta = ca.DM.zeros(4) if theta is None else theta
    physical_q = q_encoder + encoder_bias(theta)
    lengths = ca.vertcat(dh_a[0], dh_a[1] + theta[0], dh_a[2] + theta[1], *dh_a[3:])
    transforms = [ca.DM.eye(4)]
    for joint in range(6):
        transforms.append(
            transforms[-1] @ dh_transform(
                physical_q[joint], dh_d[joint], lengths[joint], dh_alpha[joint]
            )
        )
    return transforms


def forward_kinematics(q_encoder, theta=None):
    return link_transforms(q_encoder, theta)[-1]


def frame_origins(q_encoder, theta=None):
    return ca.horzcat(*[transform[:3, 3] for transform in link_transforms(q_encoder, theta)])


q_symbol = ca.SX.sym("q_encoder", 6)
theta_symbol = ca.SX.sym("theta", 4)
pose_map = ca.Function("pose", [q_symbol, theta_symbol], [forward_kinematics(q_symbol, theta_symbol)])
position_map = ca.Function("position", [q_symbol, theta_symbol], [pose_map(q_symbol, theta_symbol)[:3, 3]])
origins_map = ca.Function("origins", [q_symbol, theta_symbol], [frame_origins(q_symbol, theta_symbol)])
nominal_theta = np.zeros(4)

### Create independent pose measurements and reserve validation data

A synthetic tracker observes the complete tool pose at 36 diverse, known encoder
configurations. The simulated plant has 4 mm and −3 mm link corrections and small
joint-2/3 zero offsets. **Only data generation sees `theta_truth`.** The estimator
starts at nominal parameters. Tracker noise is 0.5 mm per Cartesian component
and 0.002 rad per rotation-vector component.

Twenty-four randomly selected poses train the estimator; the remaining twelve
are held out. The split and random seed stay fixed when changing noise, allowing
fair comparisons. Tool/base extrinsics, other DH constants, and encoder noise
during this static calibration experiment are assumed known. Estimating every
DH entry and frame offset together would introduce ambiguity.

In [ ]:
from scipy.spatial.transform import Rotation

parameter_scale = np.array([0.01, 0.01, 0.05, 0.05])
theta_truth = np.array([0.004, -0.003, 0.015, -0.012])
sigma_position = 0.0005
sigma_rotation = 0.002


def calibration_dataset(noise_multiplier=1.0):
    rng = np.random.default_rng(2026)
    configurations = rng.uniform(
        [-1.5, -1.5, 0.2, -1.0, 0.4, -1.5],
        [1.5, -0.2, 1.7, 1.0, 1.6, 1.5], (36, 6),
    )
    measured = np.array([np.asarray(pose_map(q, theta_truth)) for q in configurations])
    measured[:, :3, 3] += rng.normal(0, sigma_position * noise_multiplier, (36, 3))
    for transform in measured:
        rotation_noise = Rotation.from_rotvec(rng.normal(0, sigma_rotation * noise_multiplier, 3)).as_matrix()
        transform[:3, :3] = transform[:3, :3] @ rotation_noise
    indices = rng.permutation(36)
    return {
        "q": configurations, "poses": measured,
        "training": indices[:24], "validation": indices[24:],
        "noise_multiplier": noise_multiplier,
    }


calibration_data = calibration_dataset()
print("Training / validation poses:", len(calibration_data["training"]), "/", len(calibration_data["validation"]))

### Formulate and solve a scaled nonlinear least-squares calibration

With $\theta=D z$, $D=\operatorname{diag}(0.01\,\mathrm m,0.01\,\mathrm m,
0.05\,\mathrm{rad},0.05\,\mathrm{rad})$, solve

$$\min_{-1\le z\le1}\sum_{j\in\mathcal T}\left\|
\begin{bmatrix}
(p(q_j;Dz)-p_j)/\sigma_p\\
\operatorname{vec}(R_j^TR(q_j;Dz)-I)/(\sqrt2\sigma_R)
\end{bmatrix}\right\|_2^2.$$

The rotation term is a smooth **chordal rotation residual**:
$\|R_j^TR-I\|_F^2/2=4\sin^2(\phi/2)\approx\phi^2$ for small relative angle
$\phi$. Its nine entries encode three rotation degrees of freedom; this is a
weighted least-squares teaching objective, not an exact independent-component likelihood.
Small, bounded corrections keep the problem close to the measured frame convention.
The $12|\mathcal T|\times4$ residual Jacobian is differentiated with respect to
scaled parameters so length and angle columns can be compared sensibly.

In [ ]:
def calibration_residual(q_observed, pose_observed, theta, noise_multiplier=1.0):
    prediction = pose_map(q_observed, theta)
    position_residual = (prediction[:3, 3] - pose_observed[:3, 3]) / (sigma_position * noise_multiplier)
    relative_rotation = pose_observed[:3, :3].T @ prediction[:3, :3]
    rotation_residual = ca.vec(relative_rotation - ca.DM.eye(3)) / (np.sqrt(2) * sigma_rotation * noise_multiplier)
    return ca.vertcat(position_residual, rotation_residual)


def fit_calibration(data, training_count=24):
    problem = ca.Opti()
    normalized_theta = problem.variable(4)
    theta = ca.DM(parameter_scale) * normalized_theta
    residual = ca.vertcat(*[
        calibration_residual(data["q"][index], ca.DM(data["poses"][index]), theta, data["noise_multiplier"])
        for index in data["training"][:training_count]
    ])
    problem.minimize(ca.sumsqr(residual))
    problem.subject_to(problem.bounded(-1, normalized_theta, 1))
    problem.set_initial(normalized_theta, 0)
    problem.solver("ipopt", {"print_time": False}, {"print_level": 0, "sb": "yes", "max_iter": 100, "tol": 1e-9})
    solution = problem.solve()
    estimated_theta = np.asarray(solution.value(theta)).ravel()
    sensitivity = ca.Function("calibration_sensitivity", [normalized_theta], [ca.jacobian(residual, normalized_theta)])
    singular_values = np.linalg.svd(np.asarray(sensitivity(estimated_theta / parameter_scale)), compute_uv=False)
    return {"theta": estimated_theta, "singular_values": singular_values, "stats": solution.stats(), "training_count": training_count}


calibration_result = fit_calibration(calibration_data)
theta_estimate = calibration_result["theta"]
print("[delta a2 [m], delta a3 [m], delta2 [rad], delta3 [rad]]")
print("estimated:", theta_estimate)
print("truth (evaluation only):", theta_truth)

### Validate prediction and identifiability on data the solver did not see

Report position and rotation-angle RMSE independently on training and held-out
poses. Good training loss alone can conceal a poor model. The scaled sensitivity
must have four non-negligible singular values; otherwise some parameter combination
cannot be resolved by these poses. The default held-out position error should
fall from roughly 10 mm to roughly 1 mm, near the synthetic noise level.

Parameter-recovery assertions below use simulation truth **only after fitting**.
Real calibration must instead rely on held-out measurements, uncertainty analysis,
and independent geometric checks. Bounds alone do not make a parameter identifiable.

In [ ]:
def pose_errors(data, theta, indices):
    position_errors, rotation_errors = [], []
    for index in indices:
        prediction = np.asarray(pose_map(data["q"][index], theta))
        measurement = data["poses"][index]
        position_errors.append(np.linalg.norm(prediction[:3, 3] - measurement[:3, 3]))
        rotation_errors.append(Rotation.from_matrix(measurement[:3, :3].T @ prediction[:3, :3]).magnitude())
    return np.asarray(position_errors), np.asarray(rotation_errors)


def summarize_calibration(data, result):
    metrics = {}
    for split in ("training", "validation"):
        indices = data[split][:result["training_count"]] if split == "training" else data[split]
        for label, theta in (("nominal", nominal_theta), ("calibrated", result["theta"])):
            position_error, rotation_error = pose_errors(data, theta, indices)
            metrics[split, label] = (np.sqrt(np.mean(position_error**2)), np.sqrt(np.mean(rotation_error**2)))
            print(f"{split:10s} {label:10s}: position RMSE {metrics[split, label][0] * 1000:.3f} mm; orientation RMSE {metrics[split, label][1]:.5f} rad")
    singular_values = result["singular_values"]
    print("Scaled sensitivity singular values:", singular_values)
    print("Scaled sensitivity condition number:", singular_values[0] / singular_values[-1])
    return metrics


calibration_metrics = summarize_calibration(calibration_data, calibration_result)
assert calibration_result["stats"]["success"]
assert calibration_result["singular_values"][-1] > 1e-6 * calibration_result["singular_values"][0]
assert calibration_metrics["validation", "calibrated"][0] < 0.002
assert calibration_metrics["validation", "calibrated"][0] < calibration_metrics["validation", "nominal"][0]
np.testing.assert_allclose(theta_estimate[:2], theta_truth[:2], atol=0.001)
np.testing.assert_allclose(theta_estimate[2:], theta_truth[2:], atol=0.002)

### Derive the acceleration-controlled dynamics and discrete step

The state is $x=[q^T,v^T]^T\in\mathbb R^{12}$ in encoder coordinates;
the input is commanded joint acceleration $u\in\mathbb R^6$. Constant zero offsets
do not change $\dot q=v$. For piecewise constant input over $h=0.2$ s,

$$\dot q=v,\quad\dot v=u,\qquad
x_{k+1}=A_dx_k+B_du_k,\quad
A_d=\begin{bmatrix}I&hI\\0&I\end{bmatrix},\quad
B_d=\begin{bmatrix}\tfrac12h^2I\\hI\end{bmatrix}.$$

This discretization is exact for this surrogate. The model contains no masses,
gravity, torques, drive dynamics, or contact. Physical angle limits must be shifted
by the estimated encoder bias before constraining encoder states.

In [ ]:
sample_time = 0.2


def discrete_matrices(step):
    identity = np.eye(6)
    a_matrix = np.block([[identity, step * identity], [np.zeros((6, 6)), identity]])
    b_matrix = np.vstack([0.5 * step**2 * identity, step * identity])
    return a_matrix, b_matrix


def exact_step(state, acceleration, step=sample_time):
    return ca.vertcat(
        state[:6] + step * state[6:] + 0.5 * step**2 * acceleration,
        state[6:] + step * acceleration,
    )


a_discrete, b_discrete = discrete_matrices(sample_time)
x_symbol = ca.SX.sym("state", 12)
u_symbol = ca.SX.sym("acceleration", 6)
step_map = ca.Function("exact_double_integrator", [x_symbol, u_symbol], [exact_step(x_symbol, u_symbol)])
np.testing.assert_allclose(step_map(np.zeros(12), np.ones(6)), np.r_[np.full(6, 0.5 * sample_time**2), np.full(6, sample_time)][:, None])
q_min_encoder = q_min_physical - np.asarray(encoder_bias(theta_estimate)).ravel()
q_max_encoder = q_max_physical - np.asarray(encoder_bias(theta_estimate)).ravel()

### Reject an invalid solve before using an estimate or applying a command

Solver termination is one check. A candidate must also contain finite decisions
and satisfy every original Opti constraint within $10^{-5}$ in that row's units.
The helpers below measure wall time, retain status and iteration count, and check
$\max(0,l_g-g,g-u_g)$. On failure, print the worst rows and `g_describe` source
locations, then stop. Later cells independently check dynamics and input bounds
before a predicted acceleration reaches the simulated plant.

Residual units differ between constraints; this compact maximum is a rejection
gate, not a physically normalized accuracy score. Small equality residuals or
successful status alone cannot establish a useful or collision-free trajectory.

In [ ]:
def constraint_diagnostics(problem, candidate):
    values = candidate.value(problem.g)
    lower = candidate.value(problem.lbg)
    upper = candidate.value(problem.ubg)
    violation = max_bound_violation(values, lower, upper)
    return violation, bound_report(values, lower, upper, labels=getattr(problem, "tutorial_constraint_labels", None))


def checked_solve(problem, label, tolerance=1e-5):
    started = perf_counter()
    try:
        solution = problem.solve()
    except RuntimeError:
        print(label, "failed:", solver_summary(problem.stats(), elapsed_s=perf_counter() - started))
        violation, rows = constraint_diagnostics(problem, problem.debug)
        for row in sorted(rows, key=lambda item: item["violation"], reverse=True)[:3]:
            print("Constraint diagnostic:", row)
            print(problem.debug.g_describe(row["index"]))
        raise
    report = solver_summary(solution.stats(), elapsed_s=perf_counter() - started)
    report["constraint_violation"], _ = constraint_diagnostics(problem, solution)
    report["finite"] = bool(np.isfinite(np.asarray(solution.value(problem.x))).all())
    report["accepted"] = bool(report["success"] and report["finite"] and report["constraint_violation"] <= tolerance)
    if not report["accepted"]:
        print(label, "candidate rejected:", report)
        raise RuntimeError(label + " did not pass the numerical acceptance checks")
    return solution, report

### Formulate moving-horizon state estimation from encoders and a tool tracker

Over $L=6$ intervals, estimate seven 12-state vectors and six 6-axis acceleration
disturbances: **120 decision variables**. The known commands are $u_i$; noisy
observations are $y^q_i=q_i+\epsilon^q_i$ and
$y^p_i=p(q_i;\hat\theta)+\epsilon^p_i$. Solve

$$\min_{X,W}\|x_0-\bar x_0\|_{P^{-1}}^2+
\sum_{i=0}^{L}\left(\left\|\frac{q_i-y_i^q}{\sigma_q}\right\|^2+
\left\|\frac{p(q_i;\hat\theta)-y_i^p}{\sigma_p}\right\|^2\right)
+\sum_{i=0}^{L-1}\left\|\frac{w_i}{\sigma_a}\right\|^2,$$
$$\text{subject to }x_{i+1}=A_dx_i+B_d(u_i+w_i).$$

Use $\sigma_q=0.003$ rad, $\sigma_p=0.001$ m, and $\sigma_a=0.08$ rad/s².
The arrival standard deviations are 0.01 rad and 0.08 rad/s. The prior summarizes
earlier history approximately; a fixed diagonal arrival cost is not a full
covariance marginalization. Position observations over time and the dynamics
identify velocity; a single encoder observation does not.

Each new solve shifts the previous state trajectory and predicts one extra node.
The calibrated FK enters the tool residual, while encoder offsets are never added
to encoder observations. Write `states[:6, :]` explicitly: a single CasADi slice
indexes a flattened matrix rather than selecting all six rows across time.

In [ ]:
def build_mhe(theta, window=6, encoder_sigma=0.003, tool_sigma=0.001, acceleration_sigma=0.08):
    problem = ca.Opti()
    states = problem.variable(12, window + 1)
    disturbances = problem.variable(6, window)
    inputs = problem.parameter(6, window)
    encoders = problem.parameter(6, window + 1)
    tools = problem.parameter(3, window + 1)
    arrival = problem.parameter(12)
    arrival_sigma = ca.DM([0.01] * 6 + [0.08] * 6)
    objective = ca.sumsqr((states[:, 0] - arrival) / arrival_sigma)
    objective += ca.sumsqr((states[:6, :] - encoders) / encoder_sigma)
    objective += ca.sumsqr(disturbances / acceleration_sigma)
    for index in range(window + 1):
        objective += ca.sumsqr((position_map(states[:6, index], theta) - tools[:, index]) / tool_sigma)
    for index in range(window):
        problem.subject_to(states[:, index + 1] == exact_step(states[:, index], inputs[:, index] + disturbances[:, index]))
    problem.tutorial_constraint_labels = [f"MHE dynamics interval {index}, state {component}" for index in range(window) for component in range(12)]
    problem.minimize(objective)
    problem.solver("ipopt", {"print_time": False}, {"print_level": 0, "sb": "yes", "max_iter": 100, "tol": 1e-7})
    return {"opti": problem, "X": states, "W": disturbances, "U": inputs, "Yq": encoders, "Yp": tools, "arrival": arrival, "window": window}


def estimate_window(estimator, inputs, encoders, tools, arrival, previous_states=None):
    problem = estimator["opti"]
    problem.set_value(estimator["U"], inputs)
    problem.set_value(estimator["Yq"], encoders)
    problem.set_value(estimator["Yp"], tools)
    problem.set_value(estimator["arrival"], arrival)
    if previous_states is None:
        initial_states = np.vstack([encoders, np.zeros_like(encoders)])
    else:
        predicted_end = a_discrete @ previous_states[:, -1] + b_discrete @ inputs[:, -1]
        initial_states = np.column_stack([previous_states[:, 1:], predicted_end])
    problem.set_initial(estimator["X"], initial_states)
    problem.set_initial(estimator["W"], 0)
    solution, report = checked_solve(problem, "MHE")
    states = solution.value(estimator["X"])
    disturbances = solution.value(estimator["W"])
    defects = states[:, 1:] - a_discrete @ states[:, :-1] - b_discrete @ (inputs + disturbances)
    report["dynamics_defect"] = float(np.max(np.abs(defects)))
    if report["dynamics_defect"] > 1e-5:
        raise RuntimeError("MHE dynamics check failed before feeding MPC")
    return states[:, -1], states, report

### Test the estimator against a separate disturbed plant

The plant uses true geometry and random acceleration disturbances; the estimator
uses **fitted** calibration parameters and receives only commanded accelerations,
encoder samples, and tracker samples. Truth is retained solely for scoring.
The first six intervals fill the observation window; reported estimates start
after that warm-up rather than inventing a full history at the first sample.

The baseline velocity is the backward encoder difference
$(y^q_k-y^q_{k-1})/h$. Differencing amplifies measurement noise and also represents
an interval-average velocity, while MHE estimates the current endpoint velocity.
The default should improve velocity RMSE substantially. A longer window or a
stronger arrival prior is not automatically better when the process model is wrong.

In [ ]:
def state_estimation_data(noise_multiplier=1.0, intervals=24):
    rng = np.random.default_rng(42)
    commands = np.array([0.35 * np.sin(np.arange(intervals) * 0.25 + joint * 0.6) for joint in range(6)])
    states = np.zeros((12, intervals + 1))
    states[:6, 0] = [0.1, -0.5, 0.6, -0.2, 0.7, 0.1]
    states[6:, 0] = [0.1, -0.1, 0, 0, 0, 0]
    for index in range(intervals):
        disturbance = rng.normal(0, 0.08, 6)
        states[:, index + 1] = a_discrete @ states[:, index] + b_discrete @ (commands[:, index] + disturbance)
    encoders = states[:6] + rng.normal(0, 0.003 * noise_multiplier, (6, intervals + 1))
    tools = np.array([np.asarray(position_map(q, theta_truth)).ravel() for q in states[:6].T]).T
    tools += rng.normal(0, 0.001 * noise_multiplier, tools.shape)
    return {"truth": states, "u": commands, "q_measured": encoders, "p_measured": tools}


def run_state_estimation(data, window=6, noise_multiplier=1.0):
    estimator = build_mhe(theta_estimate, window, 0.003 * noise_multiplier, 0.001 * noise_multiplier)
    arrival = np.r_[data["q_measured"][:, 0], np.zeros(6)]
    previous_states = None
    estimates, statuses = [], []
    for index in range(window, data["truth"].shape[1]):
        start = index - window
        estimate, fitted_states, stats = estimate_window(
            estimator, data["u"][:, start:index],
            data["q_measured"][:, start:index + 1], data["p_measured"][:, start:index + 1],
            arrival, previous_states,
        )
        estimates.append(estimate)
        statuses.append(stats["success"])
        arrival, previous_states = fitted_states[:, 1], fitted_states
    estimates = np.asarray(estimates).T
    truth = data["truth"][:, window:]
    raw_velocity = np.diff(data["q_measured"], axis=1)[:, window - 1:] / sample_time
    return {
        "estimates": estimates, "truth": truth, "raw_velocity": raw_velocity,
        "time": sample_time * np.arange(window, data["truth"].shape[1]),
        "position_rmse": np.sqrt(np.mean((estimates[:6] - truth[:6])**2)),
        "velocity_rmse": np.sqrt(np.mean((estimates[6:] - truth[6:])**2)),
        "raw_velocity_rmse": np.sqrt(np.mean((raw_velocity - truth[6:])**2)),
        "success": all(statuses), "window": window,
    }


state_data = state_estimation_data()
state_result = run_state_estimation(state_data)
print("Encoder-coordinate position RMSE [rad]:", state_result["position_rmse"])
print("Velocity RMSE: finite difference / MHE [rad/s]:", state_result["raw_velocity_rmse"], "/", state_result["velocity_rmse"])
assert state_result["success"] and np.isfinite(state_result["estimates"]).all()
assert state_result["velocity_rmse"] < state_result["raw_velocity_rmse"]

### Compare measured, estimated, and true states without rerunning the solver

Start with joint 2, where the robot moves visibly. The position plot overlays
encoder observations with truth and MHE; the velocity plot contrasts noisy
differencing with the dynamic estimate. Slider changes select a joint from
cached results. To change noise or window length, use the separate experiment
button below. A delayed or oversmoothed estimate is visible even when aggregate
RMSE looks acceptable.

In [ ]:
def show_estimation_joint(joint=2):
    index = int(joint) - 1
    start = state_result["window"]
    display_plot(plot_series(state_result["time"], {
        "truth": state_result["truth"][index],
        "encoder": state_data["q_measured"][index, start:],
        "MHE": state_result["estimates"][index],
    }, title=f"Joint {joint}: position estimation", ylabel="q_encoder [rad]"))
    display_plot(plot_series(state_result["time"], {
        "truth": state_result["truth"][6 + index],
        "backward difference": state_result["raw_velocity"][index],
        "MHE": state_result["estimates"][6 + index],
    }, title=f"Joint {joint}: velocity estimation", ylabel="v [rad/s]"))


show_estimation_joint()
joint_control = widgets.IntSlider(value=2, min=1, max=6, description="Joint") if widgets else 2
experiment(show_estimation_joint, {"joint": joint_control}, title="Inspect another cached joint")

### Rerun a controlled estimation experiment

Adjust the measurement-noise multiplier and horizon length, then press **Run**.
Sensor weights change with the declared noise; process noise stays fixed. The
callback regenerates the same plant trajectory and noise draws at the selected
amplitude, enabling a useful comparison. Look for changes in velocity RMSE,
short-window noise, and long-window model bias.

In [ ]:
def estimation_experiment(noise_multiplier=1.0, window=6):
    data = state_estimation_data(float(noise_multiplier))
    result = run_state_estimation(data, int(window), float(noise_multiplier))
    print("Velocity RMSE: backward difference / MHE:", result["raw_velocity_rmse"], "/", result["velocity_rmse"])
    display_plot(plot_series(result["time"], {"true joint-2 velocity": result["truth"][7], "MHE": result["estimates"][7], "backward difference": result["raw_velocity"][1]}, title="Estimation experiment", ylabel="v2 [rad/s]"))


controls = {
    "noise_multiplier": widgets.FloatSlider(value=1, min=0.5, max=3, step=0.5, description="Noise ×", continuous_update=False) if widgets else 1.0,
    "window": widgets.IntSlider(value=6, min=3, max=10, description="Intervals", continuous_update=False) if widgets else 6,
}
experiment(estimation_experiment, controls, title="Run a new MHE experiment")

### Interpret and debug estimation failures

Poor residual fit with successful solver status can mean wrong calibration,
wrong units, underestimated process noise, or an inadequate acceleration model.
Inspect encoder and tool residuals separately. An arrival cost that is too
strong can preserve a wrong initial velocity. If IPOPT fails, inspect the
last iterate and constraint residuals using [18](18_solver_debugging.ipynb).
The MHE covariance assumptions describe this synthetic example; fitting the
residuals does not establish estimator consistency for a physical robot.